# Grid search

Settings such as `C` and `gamma` change a Kernel SVM a lot. **Grid search** tries every combination you list and keeps the one with the best cross-validated score.

This notebook goes with the **Grid search** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

## 1. Load the data

Each row is a tumour sample from a breast biopsy. The columns are eight measurements of the cell nuclei (size, texture, shape). The goal: predict whether the sample is **malignant** (`diagnosis = 1`) or **benign** (`0`).

The data is the Breast Cancer Wisconsin (Diagnostic) data set from the UCI repository (CC BY 4.0), reduced to eight columns. This is a teaching example, not medical advice.

In [ ]:
import numpy as np
import pandas as pd

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/breast_cancer.csv"
dataset = pd.read_csv(DATA_URL)
X = dataset.drop(columns="diagnosis")
y = dataset["diagnosis"]
print(dataset.shape)
dataset.head()

## 2. Split off a test set

Grid search uses cross-validation *inside* the training data. The test set stays untouched until the very end.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)

## 3. Run the search

Every cell of the grid is a 5-fold cross-validation. Here that is 5 values of `C` times 5 values of `gamma` times 5 folds = 125 models. The scaler is part of the pipeline, so it is fitted inside each fold.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipe = Pipeline([("scale", StandardScaler()), ("svm", SVC(kernel="rbf"))])
grid = {
    "svm__C": [0.1, 1, 10, 100, 1000],
    "svm__gamma": [0.001, 0.01, 0.1, 1, 10],
}
search = GridSearchCV(pipe, grid, cv=5, return_train_score=True)
search.fit(X_train, y_train)

print("best settings:", search.best_params_)
print(f"best CV accuracy: {search.best_score_:.3f}")

## 4. Draw the heatmap

This is the same picture as on the lesson page: mean CV accuracy for every cell.

In [ ]:
import matplotlib.pyplot as plt

res = pd.DataFrame(search.cv_results_)
cv_map = res.pivot(index="param_svm__C", columns="param_svm__gamma", values="mean_test_score")
fig, ax = plt.subplots(figsize=(6, 4))
im = ax.imshow(cv_map.values, cmap="viridis")
ax.set_xticks(range(len(cv_map.columns)), cv_map.columns)
ax.set_yticks(range(len(cv_map.index)), cv_map.index)
ax.set_xlabel("gamma")
ax.set_ylabel("C")
fig.colorbar(im, label="mean CV accuracy")
plt.show()

## 5. The overfitting gap

Compare the training score with the CV score for every cell. A big gap means the model memorised the training folds.

In [ ]:
res["gap"] = res["mean_train_score"] - res["mean_test_score"]
res.sort_values("gap", ascending=False)[["param_svm__C", "param_svm__gamma", "mean_train_score", "mean_test_score", "gap"]].head(5)

## 6. Score the winner once on the test set

`GridSearchCV` refits the best settings on the whole training set, so `search` can predict directly.

In [ ]:
print(f"test accuracy: {search.score(X_test, y_test):.3f}")

## 7. Try it yourself

- Add `"svm__kernel": ["linear", "rbf"]` to the grid.
- Try `RandomizedSearchCV`, which samples cells instead of trying all of them. It is much faster for big grids.